In [1]:
import numpy as np
import matplotlib.pyplot as plt
import os
import open3d as o3d
from PIL import Image


def get_cuboid_mesh(verts, elems, values, cmap="viridis"):
    
    tri_idx = np.array([[0,0,0,0,0,0,6,6,6,6,6,6],
                        [2,3,7,4,1,5,2,3,1,5,4,7],
                        [1,2,3,7,5,4,3,7,2,1,5,4]], dtype=int)
    mesh = o3d.geometry.TriangleMesh()
    tris = np.zeros([12*elems.shape[0],3], dtype=int)
    for i, elem in enumerate(elems):
        tris[i*12:(i+1)*12,:] = elem[tri_idx].T-1
    
    mesh.vertices = o3d.utility.Vector3dVector(verts)
    mesh.triangles = o3d.utility.Vector3iVector(tris)

    cmap = plt.get_cmap(cmap)
    lb, ub = min(values), max(values)
    values = (values - lb) / (ub - lb)
    mesh.vertex_colors = o3d.utility.Vector3dVector(cmap(values)[:,:3])
    return mesh
                          
data_path = r"C:\Users\XuanLiang\Documents\Netfabb_Fusion360_data\TEST\data\pstress\\"
size = os.listdir(data_path)
data_files = os.listdir(data_path)
dir_data_path = r"C:\Users\XuanLiang\Documents\Netfabb_Fusion360_data\TEST\data\pstressdir\\"
dir_data_files = os.listdir(dir_data_path)

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [6]:
for i in range(len(size)-1):
    directory = r"C:\Users\XuanLiang\Documents\Netfabb_Fusion360_data\TEST\data\pstress\images"
    data = np.load(data_path + data_files[i])
    dir_data = np.load(dir_data_path + dir_data_files[i])
    p_stress = data['pstress']
    p_stress_dir = dir_data['pstressdir']
    pstress_combo = p_stress_dir * p_stress
    verts, elems= data["verts"], data["elems"]
    mesh = get_cuboid_mesh(verts, elems, pstress_combo[:,2])
    def rotate_shape():
        rotation_angle = -np.pi / 4
        rotation_matrix_z = np.array([[np.cos(rotation_angle), -np.sin(rotation_angle), 0],
                                    [np.sin(rotation_angle), np.cos(rotation_angle), 0],
                                    [0, 0, 1]])
        rotation_matrix_x = np.array([[1, 0, 0],
                                        [0, np.cos(rotation_angle), -np.sin(rotation_angle)],
                                        [0, np.sin(rotation_angle), np.cos(rotation_angle)]])
        mesh.rotate(rotation_matrix_z, center=mesh.get_center())
        mesh.rotate(rotation_matrix_x, center=mesh.get_center())
        
    def modify_FOV(vis):
        screenshot = vis.capture_screen_float_buffer(False)
        # Convert the NumPy array to an image
        img = Image.fromarray((np.asarray(screenshot) * 255).astype('uint8'), 'RGB')
        filepath = os.path.join(directory, f'part{i}_position{j}.jpg')
        img.save(filepath)
        vis.destroy_window()
    for j in range(6):
        vis = o3d.visualization.Visualizer()
        vis.create_window()
        vis.add_geometry(mesh)
        rotate_shape()
        vis.register_animation_callback(modify_FOV)

        vis.run()
        vis.destroy_window()

NpzFile 'C:\\Users\\XuanLiang\\Documents\\Netfabb_Fusion360_data\\TEST\\data\\pstressdir\\\\23001_234eee16_1_pstressdir.npz' with keys: verts, elems, pstressdir
NpzFile 'C:\\Users\\XuanLiang\\Documents\\Netfabb_Fusion360_data\\TEST\\data\\pstressdir\\\\23018_63087a41_28_pstressdir.npz' with keys: verts, elems, pstressdir
NpzFile 'C:\\Users\\XuanLiang\\Documents\\Netfabb_Fusion360_data\\TEST\\data\\pstressdir\\\\23018_63087a41_31_pstressdir.npz' with keys: verts, elems, pstressdir
NpzFile 'C:\\Users\\XuanLiang\\Documents\\Netfabb_Fusion360_data\\TEST\\data\\pstressdir\\\\23018_63087a41_33_pstressdir.npz' with keys: verts, elems, pstressdir
NpzFile 'C:\\Users\\XuanLiang\\Documents\\Netfabb_Fusion360_data\\TEST\\data\\pstressdir\\\\23018_63087a41_34_pstressdir.npz' with keys: verts, elems, pstressdir
